# Wednesday - Sales Data Cleaning

This notebook cleans the original sales dataset step by step.

The original raw dataset is preserved, and a separate working copy is used for cleaning.

### Sales Data Cleaning Log

| Cleaning Step | Action Taken | Reason |
|---|---|---|
| Remove duplicates | Removed 40 duplicate rows | Avoid counting duplicate records |
| Standardize labels | Standardized 16 region labels into 4 regions and 12 status labels into 3 statuses | Ensure consistent categories |
| Convert dates | Handled 714 YYYY-MM-DD, 213 DD/MM/YYYY, and 113 Month DD, YYYY date values| Enable date analysis |
| Clean prices | Removed dollar signs and commas from 297 unit_price values and converted them to numeric values| Enable numerical calculations |
| Clean quantities | Removed 43 rows (30 missing and 13 invalid quantities) | Avoid unreliable revenue calculations |
| Fill missing values | Filled 25 missing customer values with "Unknown" and 60 missing discount values with 0.0 | Use suitable default values |
| Calculate revenue | Added revenue column using quantity, unit_price, and discount; total revenue: 4,162,140.15 | Calculate sales revenue |

**Why were invalid quantity rows dropped instead of filled?**

Quantity represents the actual number of products sold. Guessing missing or invalid quantities could produce incorrect revenue. Filling is more suitable when a reasonable default is available, such as filling a missing discount with 0.0.


### Loading the Original Dataset

Load the original sales CSV file into a DataFrame and create a separate working copy. All cleaning operations will be performed on the working copy to keep the original raw dataset unchanged.


In [1]:
import pandas as pd

# Load original sales data
df_raw = pd.read_csv("data/raw/sales.csv")

# Create a separate working copy
df = df_raw.copy()

# Check original and working data
print("Original dataset shape:", df_raw.shape)
print("Working dataset shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())

Original dataset shape: (1040, 10)
Working dataset shape: (1040, 10)

First 5 rows:


,order_id,order_date,customer,region,product,category,quantity,unit_price,discount,status
0,1010,2026-05-01,CUST-0035,East,Filing Cabinet,Furniture,4.0,249.00,0.10,cancelled
1,1286,"Apr 11, 2026",CUST-0046,west,Docking Station,Electronics,13.0,219.00,0.00,Cancelled
2,1675,13/02/2026,CUST-0136,West,Office Chair,Furniture,20.0,1099.00,0.00,Pending
3,1100,06/03/2026,CUST-0011,East,Office Chair,Furniture,2.0,"$1,099.00",0.15,Shipped
4,1668,2026-05-12,CUST-0039,East,Filing Cabinet,Furniture,6.0,$249.00,0.05,Pending



Missing values:
order_id       0
order_date     0
customer      25
region         0
product        0
category       0
quantity      32
unit_price     0
discount      62
status         0
dtype: int64



### Step 1: Remove Exact Duplicate Rows

Remove exact duplicate rows from the working DataFrame to ensure that the same sales transaction is not counted more than once during data analysis.


In [2]:
# Count duplicate rows before removing them
duplicates_before = df.duplicated().sum()

print("Duplicate rows before cleaning:", duplicates_before)

# Remove exact duplicate rows
df = df.drop_duplicates()

# Check dataset after removing duplicates
print("Dataset shape after removing duplicates:", df.shape)
print("Duplicate rows after cleaning:", df.duplicated().sum())

Duplicate rows before cleaning: 40
Dataset shape after removing duplicates: (1000, 10)
Duplicate rows after cleaning: 0



### Step 2: Standardize Region and Status Labels

Remove extra spaces and apply title case to the `region` and `status` columns. This ensures consistent category names and prevents values with different capitalization or spacing from being treated as separate categories.


In [3]:
# Standardize region names
df["region"] = df["region"].str.strip().str.title()

# Standardize status names
df["status"] = df["status"].str.strip().str.title()

# Check results
print("Regions after cleaning:")
print(sorted(df["region"].unique()))

print("\nStatuses after cleaning:")
print(sorted(df["status"].unique()))

Regions after cleaning:
['East', 'North', 'South', 'West']

Statuses after cleaning:
['Cancelled', 'Pending', 'Shipped']



### Step 3: Parse Order Dates

Convert the `order_date` column into proper datetime values by explicitly handling all three date formats: YYYY-MM-DD, DD/MM/YYYY, and Month DD, YYYY. This ensures that dates are interpreted correctly and consistently.


In [4]:
# Convert the three date formats separately

date_format_1 = pd.to_datetime(df["order_date"], format="%Y-%m-%d", errors="coerce")

date_format_2 = pd.to_datetime(df["order_date"], format="%d/%m/%Y", errors="coerce")

date_format_3 = pd.to_datetime(df["order_date"], format="%b %d, %Y", errors="coerce")

# Combine successfully converted dates
df["order_date"] = date_format_1.fillna(date_format_2).fillna(date_format_3)

# Check the results
print("Missing dates after conversion:", df["order_date"].isna().sum())

print("\nFirst 10 cleaned dates:")
print(df["order_date"].head(10))

Missing dates after conversion: 0

First 10 cleaned dates:
0   2026-05-01
1   2026-04-11
2   2026-02-13
3   2026-03-06
4   2026-05-12
5   2026-06-19
6   2026-03-14
7   2026-04-01
8   2026-03-31
9   2026-02-01
Name: order_date, dtype: datetime64[us]



### Step 4: Convert Unit Prices to Numbers

Remove dollar signs ($) and commas (,) from the `unit_price` column and convert the values into numeric format. This is necessary to perform mathematical calculations and calculate revenue correctly.


In [5]:
# Remove dollar signs and commas
df["unit_price"] = (
    df["unit_price"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

# Check cleaned prices
print("Unit price data type:", df["unit_price"].dtype)

print("\nFirst 10 cleaned prices:")
print(df["unit_price"].head(10))

print("\nMissing prices:", df["unit_price"].isna().sum())

Unit price data type: float64

First 10 cleaned prices:
0     249.0
1     219.0
2    1099.0
3    1099.0
4     249.0
5    1249.0
6     129.0
7     129.0
8      89.0
9    1249.0
Name: unit_price, dtype: float64

Missing prices: 0



### Step 5: Clean Quantity Values

Treat quantity values of zero or below, or above 1000, as missing values. Drop every row where quantity is missing and convert the remaining quantity values to integers. This prevents invalid quantities from affecting sales analysis and revenue calculations.


In [6]:
# Convert quantity to numeric
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")

# Count invalid quantities
invalid_quantity = (df["quantity"] <= 0) | (df["quantity"] > 1000)

print("Missing quantities before cleaning:", df["quantity"].isna().sum())
print("Invalid quantities (<= 0 or > 1000):", invalid_quantity.sum())

# Replace invalid quantities with missing values
df["quantity"] = df["quantity"].mask(invalid_quantity)

# Remove rows where quantity is missing
df = df.dropna(subset=["quantity"])

# Convert quantity to integer
df["quantity"] = df["quantity"].astype(int)

# Check results
print("\nDataset shape after quantity cleaning:", df.shape)
print("Missing quantities after cleaning:", df["quantity"].isna().sum())
print("Quantity data type:", df["quantity"].dtype)
print("Minimum quantity:", df["quantity"].min())
print("Maximum quantity:", df["quantity"].max())

Missing quantities before cleaning: 30
Invalid quantities (<= 0 or > 1000): 13

Dataset shape after quantity cleaning: (957, 10)
Missing quantities after cleaning: 0
Quantity data type: int64
Minimum quantity: 1
Maximum quantity: 20



### Step 6: Fill Missing Values

Fill missing values in the `discount` column with `0.0` and missing values in the `customer` column with `"Unknown"`. This preserves valid sales records while ensuring that these columns contain no missing values.


In [7]:
# Fill missing customer values
df["customer"] = df["customer"].fillna("Unknown")

# Fill missing discount values
df["discount"] = df["discount"].fillna(0.0)

# Check missing values
print("Missing values after filling:")
print(df[["customer", "discount"]].isna().sum())

print("\nUnknown customers:", (df["customer"] == "Unknown").sum())
print("Discount data type:", df["discount"].dtype)

Missing values after filling:
customer    0
discount    0
dtype: int64

Unknown customers: 24
Discount data type: float64



### Step 7: Calculate Revenue

Create a new `revenue` column by multiplying `quantity` by `unit_price` and `(1 - discount)`, then round the result to 2 decimal places. This calculates the final revenue for each sales transaction after applying the discount.


In [8]:
# Calculate revenue after discount
df["revenue"] = (df["quantity"] * df["unit_price"] * (1 - df["discount"])).round(2)

# Check the new revenue column
print("First 10 revenue values:")
print(df["revenue"].head(10))

print("\nTotal revenue:", round(df["revenue"].sum(), 2))

print("\nDataset shape:", df.shape)

First 10 revenue values:
0      896.40
1     2847.00
2    21980.00
3     1868.30
4     1419.30
5    23731.00
6     1864.05
7      516.00
8     1157.00
9    13739.00
Name: revenue, dtype: float64

Total revenue: 4162140.15

Dataset shape: (957, 11)



### Final Validation and Save Cleaned Dataset

Validate the cleaned dataset using assertions to ensure that all cleaning requirements are satisfied. Save the final cleaned dataset as a CSV file in the processed data folder while keeping the original raw dataset unchanged.


In [9]:
# Final validation checks

print("Dataset shape:", df.shape)

print("\nMissing values in all columns:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nUnique order IDs:", df["order_id"].is_unique)

print("\nUnique regions:", df["region"].nunique())
print("Unique statuses:", df["status"].nunique())

print("\nAll dates parsed:", df["order_date"].notna().all())

print("\nTotal revenue:", round(df["revenue"].sum(), 2))


# Acceptance checks
assert len(df) == 957
assert df["order_id"].is_unique
assert df["region"].nunique() == 4
assert df["status"].nunique() == 3
assert df["order_date"].notna().all()
assert df.isna().sum().sum() == 0
assert round(df["revenue"].sum(), 2) == 4162140.15

print("\nAll validation checks passed!")

Dataset shape: (957, 11)

Missing values in all columns:
order_id      0
order_date    0
customer      0
region        0
product       0
category      0
quantity      0
unit_price    0
discount      0
status        0
revenue       0
dtype: int64

Duplicate rows: 0

Unique order IDs: True

Unique regions: 4
Unique statuses: 3

All dates parsed: True

Total revenue: 4162140.15

All validation checks passed!


In [10]:
# Save cleaned dataset
df.to_csv("data/processed/sales_clean.csv", index=False)

print("Cleaned dataset saved successfully!")

print("File location: data/processed/sales_clean.csv")

Cleaned dataset saved successfully!
File location: data/processed/sales_clean.csv


In [11]:
# Read the saved CSV file
df_check = pd.read_csv("data/processed/sales_clean.csv")

print("Saved file shape:", df_check.shape)

print("\nFirst 5 rows:")
display(df_check.head())

Saved file shape: (957, 11)

First 5 rows:


,order_id,order_date,customer,region,product,category,quantity,unit_price,discount,status,revenue
0,1010,2026-05-01,CUST-0035,East,Filing Cabinet,Furniture,4,249.0,0.10,Cancelled,896.4
1,1286,2026-04-11,CUST-0046,West,Docking Station,Electronics,13,219.0,0.00,Cancelled,2847.0
2,1675,2026-02-13,CUST-0136,West,Office Chair,Furniture,20,1099.0,0.00,Pending,21980.0
3,1100,2026-03-06,CUST-0011,East,Office Chair,Furniture,2,1099.0,0.15,Shipped,1868.3
4,1668,2026-05-12,CUST-0039,East,Filing Cabinet,Furniture,6,249.0,0.05,Pending,1419.3


In [12]:
print("FINAL CLEANING SUMMARY")
print("----------------------")

print("Dataset shape:", df.shape)
print("Total revenue:", round(df["revenue"].sum(), 2))
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Order IDs unique:", df["order_id"].is_unique)
print("Number of regions:", df["region"].nunique())
print("Number of statuses:", df["status"].nunique())
print("All dates parsed:", df["order_date"].notna().all())
print("Saved file shape:", df_check.shape)

FINAL CLEANING SUMMARY
----------------------
Dataset shape: (957, 11)
Total revenue: 4162140.15
Missing values: 0
Duplicate rows: 0
Order IDs unique: True
Number of regions: 4
Number of statuses: 3
All dates parsed: True
Saved file shape: (957, 11)


In [13]:
# Check cleaning counts from the original dataset
df_check = df_raw.drop_duplicates().copy()

print("Prices containing $ or comma:")
print(df_raw["unit_price"].astype(str).str.contains(r"[$,]", regex=True).sum())

print("\nMissing customer values after removing duplicates:")
print(df_check["customer"].isna().sum())

print("\nMissing discount values after removing duplicates:")
print(df_check["discount"].isna().sum())

Prices containing $ or comma:
297

Missing customer values after removing duplicates:
25

Missing discount values after removing duplicates:
60
